# Rice Deep-Dive: India's Dominance & the 2023 Export Ban

**Purpose:**  
The data audit (`docs/rasheed/faostat_data_audit.md`) identifies rice as the  
**strongest finding** in the dataset:

> *"India supplies 49.2% of traded milled rice. Saudi Arabia takes 75% of  
> its rice from India, and India restricted rice exports in 2023. The  
> vulnerability the analysis measures has already been demonstrated live."*

This is more compelling than the wheat story because:
1. **Concentration is higher** — one country at ~49% vs wheat's largest at ~18%
2. **The shock already happened** — India's 2023 non-basmati rice export ban  
   is a real-world proof-of-concept for the risk index
3. **The most-exposed importers are identifiable** — Saudi Arabia, UAE, Egypt,  
   Iran can be named with numbers

---

## What This Notebook Covers

1. India's share of global milled rice exports over time
2. Country-by-country exposure to India as a rice supplier
3. Pre vs post 2023 ban comparison (did countries diversify?)
4. Shannon & HHI for rice importers — who has no alternatives?
5. Headline tables and charts for the presentation

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# ---- Paths ----
ROOT = Path(".").resolve().parent
TRADE_MATRIX = ROOT / "data" / "cleaned" / "trade_matrix_cleaned.csv"
VIZ_DIR = ROOT / "visualizations"
OUTPUT_DIR = ROOT / "data" / "cleaned"
VIZ_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Rice items in FAOSTAT ----
# The trade matrix has two rice items. Both map to the "Rice" commodity.
RICE_ITEMS = ["Rice, milled", "Rice, paddy (rice milled equivalent)"]

# ---- India's FAOSTAT area code ----
INDIA_CODE = "100"  # FAOSTAT area code for India
INDIA_NAME = "India"

# ---- Year the export ban was announced (July 2023) ----
BAN_YEAR = 2023

print(f"Trade matrix: {TRADE_MATRIX}")
print(f"Exists: {TRADE_MATRIX.exists()}")

## 1. Load & Prepare Rice Trade Data

In [ ]:
# ---- Load the full trade matrix ----
raw = pd.read_csv(TRADE_MATRIX)
print(f"Total rows: {len(raw):,}")

# ---- Filter to rice, import quantities, positive values ----
rice = raw[
    (raw["Item"].isin(RICE_ITEMS))
    & (raw["Element"] == "Import quantity")
    & (raw["Unit"] == "t")
    & (raw["Value"] > 0)
].copy()

print(f"Rice import rows: {len(rice):,}")
print(f"Years: {rice['Year'].min()} - {rice['Year'].max()}")
print(f"Items: {rice['Item'].unique().tolist()}")
print(f"Reporters: {rice['Reporter Countries'].nunique()}")
print(f"Partners (exporters): {rice['Partner Countries'].nunique()}")

In [ ]:
# ---- Aggregate across rice sub-items per reporter-partner-year ----
# "Rice, milled" and "Rice, paddy (rice milled equivalent)" are summed together.
rice_flows = (
    rice.groupby(
        ["Reporter Country Code", "Reporter Countries",
         "Partner Country Code", "Partner Countries", "Year"],
        as_index=False,
    )["Value"]
    .sum()
    .rename(columns={"Value": "import_t"})
)

# ---- Total rice imports per reporter-year ----
rice_totals = (
    rice_flows.groupby(
        ["Reporter Country Code", "Reporter Countries", "Year"],
        as_index=False,
    )["import_t"]
    .sum()
    .rename(columns={"import_t": "total_import_t"})
)

# ---- Partner shares ----
rice_flows = rice_flows.merge(
    rice_totals,
    on=["Reporter Country Code", "Reporter Countries", "Year"],
    how="left",
)
rice_flows["partner_share"] = rice_flows["import_t"] / rice_flows["total_import_t"]

print(f"Rice bilateral flows: {len(rice_flows):,}")
rice_flows.head(5)

## 2. India's Global Export Share Over Time

How dominant is India in the global rice trade? We compute India's share of  
total rice imports reported by all countries, year by year.

Note: we're using **importer-reported** data (what everyone says they import  
from India), which is the more auditable side of the trade per the methodology  
spec.

In [ ]:
# ---- Global total rice imports per year (from all sources) ----
global_total = (
    rice_flows.groupby("Year", as_index=False)["import_t"].sum()
    .rename(columns={"import_t": "global_imports_t"})
)

# ---- Total imports from India per year (summed across all reporters) ----
from_india = (
    rice_flows[rice_flows["Partner Countries"] == INDIA_NAME]
    .groupby("Year", as_index=False)["import_t"]
    .sum()
    .rename(columns={"import_t": "from_india_t"})
)

india_share = global_total.merge(from_india, on="Year", how="left")
india_share["from_india_t"] = india_share["from_india_t"].fillna(0)
india_share["india_share"] = india_share["from_india_t"] / india_share["global_imports_t"]
india_share["india_mt"] = india_share["from_india_t"] / 1e6  # million tonnes

print("India's share of global rice trade (importer-reported):")
print(india_share[["Year", "global_imports_t", "from_india_t", "india_share"]].to_string(
    index=False,
    formatters={
        "global_imports_t": "{:,.0f}".format,
        "from_india_t": "{:,.0f}".format,
        "india_share": "{:.1%}".format,
    },
))

In [ ]:
# ---- Chart: India's global rice export share over time ----
fig, ax1 = plt.subplots(figsize=(12, 5))

# Bar: absolute volume (million tonnes)
ax1.bar(
    india_share["Year"], india_share["india_mt"],
    alpha=0.4, color="#1565c0", label="Volume from India (Mt)",
)
ax1.set_ylabel("Rice imports from India (million tonnes)", color="#1565c0")
ax1.set_xlabel("Year")

# Line: share
ax2 = ax1.twinx()
ax2.plot(
    india_share["Year"], india_share["india_share"] * 100,
    color="#d32f2f", marker="o", linewidth=2.5, label="India's share (%)",
)
ax2.set_ylabel("India's share of global rice trade (%)", color="#d32f2f")

# Mark the 2023 export ban
ax1.axvline(BAN_YEAR, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax1.text(
    BAN_YEAR + 0.1, ax1.get_ylim()[1] * 0.9,
    "India rice\nexport ban",
    fontsize=9, color="black", va="top",
)

ax1.set_title("India's Dominance in Global Rice Trade", fontsize=14)
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88), fontsize=9)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "india_rice_global_share.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Country-Level Exposure to India

Which importers rely most heavily on India for their rice? We compute each  
country's share of rice imports sourced from India, averaged over the most  
recent window.

In [ ]:
# ---- Compute India-share per importer per year ----
india_flows = rice_flows[rice_flows["Partner Countries"] == INDIA_NAME][
    ["Reporter Country Code", "Reporter Countries", "Year",
     "import_t", "total_import_t", "partner_share"]
].rename(columns={
    "import_t": "from_india_t",
    "partner_share": "india_share",
})

# ---- Average over a pre-ban window (2021-2022) and post-ban (2023+) ----
# Pre-ban: 2021-2022 (before the July 2023 ban)
pre_ban = india_flows[india_flows["Year"].between(2021, 2022)].copy()
pre_avg = (
    pre_ban.groupby(["Reporter Country Code", "Reporter Countries"], as_index=False)
    .agg(
        pre_india_share=("india_share", "mean"),
        pre_from_india_t=("from_india_t", "mean"),
        pre_total_t=("total_import_t", "mean"),
    )
)

# Post-ban: 2023 onwards
post_ban = india_flows[india_flows["Year"] >= 2023].copy()
post_avg = (
    post_ban.groupby(["Reporter Country Code", "Reporter Countries"], as_index=False)
    .agg(
        post_india_share=("india_share", "mean"),
        post_from_india_t=("from_india_t", "mean"),
        post_total_t=("total_import_t", "mean"),
    )
)

# ---- Merge pre and post for comparison ----
exposure = pre_avg.merge(
    post_avg,
    on=["Reporter Country Code", "Reporter Countries"],
    how="outer",
)

# Change in India share
exposure["share_change"] = exposure["post_india_share"] - exposure["pre_india_share"]
exposure["volume_change_t"] = exposure["post_from_india_t"] - exposure["pre_from_india_t"]

# Sort by pre-ban exposure
exposure = exposure.sort_values("pre_india_share", ascending=False)

print(f"Countries with India rice exposure data: {len(exposure)}")
print("\nTop 20 most India-dependent rice importers (pre-ban 2021-22 average):")
# Filter to significant importers (>=5,000 t/year)
significant = exposure[exposure["pre_total_t"] >= 5000].head(20)
print(
    significant[
        ["Reporter Countries", "pre_india_share", "post_india_share",
         "share_change", "pre_from_india_t", "pre_total_t"]
    ].to_string(
        index=False,
        formatters={
            "pre_india_share": "{:.1%}".format,
            "post_india_share": "{:.1%}".format,
            "share_change": "{:+.1%}".format,
            "pre_from_india_t": "{:,.0f}".format,
            "pre_total_t": "{:,.0f}".format,
        },
    )
)

In [ ]:
# ---- Chart: Top importers' India exposure, pre vs post ban ----
# Show the 15 most exposed (by pre-ban share) with significant volume
chart_data = exposure[exposure["pre_total_t"] >= 5000].head(15).copy()
chart_data = chart_data.sort_values("pre_india_share", ascending=True)  # for horizontal bars

fig, ax = plt.subplots(figsize=(12, 8))

y = np.arange(len(chart_data))
height = 0.35

# Pre-ban bars
bars_pre = ax.barh(
    y + height / 2, chart_data["pre_india_share"] * 100,
    height, label="Pre-ban (2021-22)", color="#1565c0", alpha=0.8,
)

# Post-ban bars (may have NaN for countries without 2023 data)
post_vals = chart_data["post_india_share"].fillna(0) * 100
bars_post = ax.barh(
    y - height / 2, post_vals,
    height, label="Post-ban (2023+)", color="#d32f2f", alpha=0.8,
)

ax.set_yticks(y)
ax.set_yticklabels(chart_data["Reporter Countries"], fontsize=9)
ax.set_xlabel("Share of rice imports from India (%)")
ax.set_title("Rice Import Dependence on India: Pre vs Post 2023 Export Ban", fontsize=13)
ax.legend(loc="lower right")
ax.axvline(50, color="grey", linestyle=":", alpha=0.5)
ax.text(51, len(chart_data) - 1, "50%", fontsize=8, color="grey")

plt.tight_layout()
plt.savefig(str(VIZ_DIR / "rice_india_exposure_pre_post_ban.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 4. Rice Supplier Concentration Over Time

Compute HHI and Shannon for rice importers year-by-year. Did concentration  
change after 2023? The Egypt/Turkiye wheat pattern showed re-concentration  
after a temporary diversification — does rice show the same?

In [ ]:
# ---- Compute HHI and Shannon per importer-year ----

def shannon_h(shares):
    """Shannon diversity: H = -sum(p * ln(p)), ignoring zeros."""
    p = shares[shares > 0].values
    return -np.sum(p * np.log(p)) if len(p) > 1 else 0.0


rice_conc = (
    rice_flows.groupby(
        ["Reporter Country Code", "Reporter Countries", "Year"],
        as_index=False,
    ).agg(
        hhi=("partner_share", lambda x: (x ** 2).sum()),
        shannon=("partner_share", shannon_h),
        partner_count=("Partner Country Code", "nunique"),
        top_share=("partner_share", "max"),
        total_t=("import_t", "sum"),
    )
)

rice_conc["eff_suppliers_hhi"] = 1.0 / rice_conc["hhi"]
rice_conc["eff_suppliers_shannon"] = np.exp(rice_conc["shannon"])

print(f"Rice concentration: {len(rice_conc):,} importer-year groups")
rice_conc.head(5)

In [ ]:
# ---- Track concentration over time for the most India-exposed importers ----
spotlight = [
    "Saudi Arabia", "Egypt", "United Arab Emirates",
    "Republic of Korea", "Singapore", "Japan",
    "Philippines", "Indonesia", "Malaysia",
]

spot_data = rice_conc[rice_conc["Reporter Countries"].isin(spotlight)].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = [
    ("hhi", "HHI (concentration)"),
    ("shannon", "Shannon H (diversity)"),
    ("eff_suppliers_hhi", "Effective suppliers (1/HHI)"),
    ("top_share", "Top partner share"),
]

for ax, (col, label) in zip(axes.flat, metrics):
    for country in spotlight:
        sub = spot_data[spot_data["Reporter Countries"] == country]
        if not sub.empty:
            ax.plot(sub["Year"], sub[col], marker=".", markersize=4, label=country)
    
    ax.axvline(BAN_YEAR, color="red", linestyle="--", alpha=0.5, linewidth=1.5)
    ax.set_ylabel(label)
    ax.set_xlabel("Year")
    ax.set_title(label)
    ax.legend(fontsize=7, ncol=2)

fig.suptitle("Rice: Supplier Concentration Over Time for Key Importers",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "rice_concentration_spotlight.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 5. What Happens If India Stops? (Rice-Specific Removal)

The supplier-removal simulation notebook covers all three commodities.  
Here we do the rice-specific version with more detail and a narrative focus.

In [ ]:
# ---- Average flows over 2021-2022 (pre-ban baseline) ----
pre_flows = (
    rice_flows[rice_flows["Year"].between(2021, 2022)]
    .groupby(
        ["Reporter Country Code", "Reporter Countries",
         "Partner Country Code", "Partner Countries"],
        as_index=False,
    ).agg(mean_import_t=("import_t", "mean"))
)

pre_totals = (
    pre_flows.groupby(
        ["Reporter Country Code", "Reporter Countries"],
        as_index=False,
    )["mean_import_t"]
    .sum()
    .rename(columns={"mean_import_t": "baseline_t"})
)

# Volume from India per importer
from_india_vol = (
    pre_flows[pre_flows["Partner Countries"] == INDIA_NAME]
    [["Reporter Country Code", "Reporter Countries", "mean_import_t"]]
    .rename(columns={"mean_import_t": "from_india_t"})
)

# Remaining suppliers if India removed
remaining = (
    pre_flows[pre_flows["Partner Countries"] != INDIA_NAME]
    .groupby(["Reporter Country Code", "Reporter Countries"], as_index=False)
    .agg(
        remaining_suppliers=("Partner Country Code", "nunique"),
        remaining_volume_t=("mean_import_t", "sum"),
    )
)

# Assemble impact table
impact = pre_totals.merge(from_india_vol, on=["Reporter Country Code", "Reporter Countries"], how="inner")
impact = impact.merge(remaining, on=["Reporter Country Code", "Reporter Countries"], how="left")
impact["remaining_suppliers"] = impact["remaining_suppliers"].fillna(0).astype(int)
impact["remaining_volume_t"] = impact["remaining_volume_t"].fillna(0)
impact["share_lost"] = impact["from_india_t"] / impact["baseline_t"]
impact["volume_gap_kt"] = impact["from_india_t"] / 1000

impact = impact.sort_values("share_lost", ascending=False)

# ---- Headline numbers ----
n_over50 = (impact["share_lost"] > 0.50).sum()
n_over25 = (impact["share_lost"] > 0.25).sum()
total_removed_mt = impact["from_india_t"].sum() / 1e6

print(f"If India stopped exporting rice (pre-ban 2021-22 baseline):")
print(f"  Countries losing >50% of rice imports: {n_over50}")
print(f"  Countries losing >25% of rice imports: {n_over25}")
print(f"  Total volume removed: {total_removed_mt:.2f} Mt")
print(f"  Countries affected: {len(impact)}")
print(f"\nTop 20 most-affected importers:")
print(
    impact.head(20)[
        ["Reporter Countries", "baseline_t", "from_india_t", "share_lost",
         "remaining_suppliers", "remaining_volume_t"]
    ].to_string(
        index=False,
        formatters={
            "baseline_t": "{:,.0f}".format,
            "from_india_t": "{:,.0f}".format,
            "share_lost": "{:.1%}".format,
            "remaining_volume_t": "{:,.0f}".format,
        },
    )
)

In [ ]:
# ---- Chart: Share lost if India removed, top 20 importers ----
chart = impact[
    impact["baseline_t"] >= 5000  # significant importers only
].head(20).copy()
chart = chart.sort_values("share_lost", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

colors = ["#d32f2f" if s > 0.5 else "#ff9800" if s > 0.25 else "#1565c0"
          for s in chart["share_lost"]]

ax.barh(chart["Reporter Countries"], chart["share_lost"] * 100, color=colors)
ax.axvline(50, color="grey", linestyle=":", alpha=0.6)
ax.axvline(25, color="grey", linestyle=":", alpha=0.4)
ax.set_xlabel("Share of rice imports lost (%)")
ax.set_title(
    f"If India Stopped Exporting Rice:\n"
    f"{n_over50} countries lose >50% (red), {n_over25} lose >25% (orange)",
    fontsize=13,
)

# Add volume annotations
for i, (_, row) in enumerate(chart.iterrows()):
    ax.text(
        row["share_lost"] * 100 + 1, i,
        f"{row['volume_gap_kt']:.0f} kt",
        va="center", fontsize=8, color="grey",
    )

plt.tight_layout()
plt.savefig(str(VIZ_DIR / "rice_india_removal_impact.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 6. Who Else Exports Rice? (Alternative Suppliers)

If India exits, who could fill the gap? This is not a prediction (that  
would require spare capacity and freight modelling), but it shows which  
countries have the scale to matter.

In [ ]:
# ---- Global rice exports by partner (mirror: what importers report) ----
global_exports = (
    rice_flows[rice_flows["Year"].between(2021, 2022)]
    .groupby(["Partner Country Code", "Partner Countries"], as_index=False)
    .agg(total_exported_t=("import_t", "sum"))
)
global_exports = global_exports.sort_values("total_exported_t", ascending=False)
global_sum = global_exports["total_exported_t"].sum()
global_exports["global_share"] = global_exports["total_exported_t"] / global_sum
global_exports["cumulative_share"] = global_exports["global_share"].cumsum()

print("Global rice exporters (importer-reported, 2021-22 total):")
print(
    global_exports.head(15)[
        ["Partner Countries", "total_exported_t", "global_share", "cumulative_share"]
    ].to_string(
        index=False,
        formatters={
            "total_exported_t": "{:,.0f}".format,
            "global_share": "{:.1%}".format,
            "cumulative_share": "{:.1%}".format,
        },
    )
)

# ---- Pie chart: top rice exporters ----
fig, ax = plt.subplots(figsize=(8, 8))
top_exp = global_exports.head(6).copy()
other = pd.DataFrame({
    "Partner Countries": ["All others"],
    "total_exported_t": [global_sum - top_exp["total_exported_t"].sum()],
})
pie_data = pd.concat([top_exp, other], ignore_index=True)

colors = ["#d32f2f"] + ["#64b5f6"] * (len(pie_data) - 2) + ["#e0e0e0"]
explode = [0.08] + [0] * (len(pie_data) - 1)

ax.pie(
    pie_data["total_exported_t"],
    labels=pie_data["Partner Countries"],
    autopct="%1.1f%%",
    colors=colors,
    explode=explode,
    startangle=90,
    pctdistance=0.8,
)
ax.set_title("Global Rice Export Shares (2021-22)\nIndia highlighted in red", fontsize=13)
plt.tight_layout()
plt.savefig(str(VIZ_DIR / "rice_global_export_shares_pie.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 7. Save Outputs & Headline Summary

In [ ]:
# ---- Save detailed outputs ----
exposure.to_csv(OUTPUT_DIR / "rice_india_exposure.csv", index=False)
impact.to_csv(OUTPUT_DIR / "rice_india_removal_impact.csv", index=False)
rice_conc.to_csv(OUTPUT_DIR / "rice_concentration_by_year.csv", index=False)

print("Saved:")
print(f"  {OUTPUT_DIR / 'rice_india_exposure.csv'}")
print(f"  {OUTPUT_DIR / 'rice_india_removal_impact.csv'}")
print(f"  {OUTPUT_DIR / 'rice_concentration_by_year.csv'}")

print("\n" + "=" * 60)
print("RICE DEEP-DIVE: HEADLINE NUMBERS")
print("=" * 60)

latest_share = india_share[india_share["Year"] == india_share["Year"].max()]
if not latest_share.empty:
    print(f"\nIndia's global rice export share (latest year): "
          f"{latest_share.iloc[0]['india_share']:.1%}")

print(f"\nPeak India share: {india_share['india_share'].max():.1%} "
      f"(year {india_share.loc[india_share['india_share'].idxmax(), 'Year']})")

# Most exposed
sig_exposure = exposure[exposure["pre_total_t"] >= 5000].head(5)
print(f"\nMost India-dependent rice importers (pre-ban, >=5kt/yr):")
for _, row in sig_exposure.iterrows():
    post = f" -> {row['post_india_share']:.0%}" if pd.notna(row['post_india_share']) else ""
    print(f"  {row['Reporter Countries']}: {row['pre_india_share']:.0%}{post}")

print(f"\nIf India stopped exporting rice:")
print(f"  {n_over50} countries lose >50% of rice imports")
print(f"  {n_over25} countries lose >25%")
print(f"  {total_removed_mt:.1f} Mt removed from trade")

print("\nVisualizations saved:")
print("  - india_rice_global_share.png")
print("  - rice_india_exposure_pre_post_ban.png")
print("  - rice_concentration_spotlight.png")
print("  - rice_india_removal_impact.png")
print("  - rice_global_export_shares_pie.png")